In [10]:
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import os
from collections import Counter
import pandas as pd
import torchvision
from PIL import Image
from torch.utils.data import Dataset

# ignore warnings
import argparse
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import wandb
from Models.logistic_regression_net import LinearClassificationNet
from scipy.io import arff
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from utils.dataset_utils import DatasetUtils
from utils.dutch import TabularDataset
from utils.model_utils import ModelUtils
from utils.tabular_datasets_utils import dataset_to_numpy, load_dutch

from FairReg.DPLUtils.regularization_config import RegularizationConfig
from FairReg.Learning.learning_new import Learning
from FairReg.Regularization.EqualizedOddsLoss import EqualizedOddsLoss
from FairReg.Regularization.RegularizationLoss import RegularizationLoss

In [11]:
class CelebaDatasetHairColor(Dataset):
    """Definition of the dataset used for the Celeba Dataset."""

    def __init__(
        self,
        csv_path: str,
        image_path: str,
        transform: torchvision.transforms = None,
        debug: bool = True,
    ) -> None:
        """Initialization of the dataset.

        Args:
        ----
            csv_path (str): path of the csv file with all the information
             about the dataset
            image_path (str): path of the images
            transform (torchvision.transforms, optional): Transformation to apply
            to the images. Defaults to None.
        """
        dataframe = pd.read_csv(csv_path)

        targets = [item for item in dataframe["Hair_Color"].tolist()]
        self.targets = targets
        self.sensitive_attributes = dataframe["Gender"].tolist()
        self.samples = list(dataframe["image_id"])
        self.n_samples = len(dataframe)
        self.transform = transform
        self.image_path = image_path
        self.debug = debug
        self.indexes = range(len(self.samples))

        if not self.debug:
            self.images = [
                Image.open(os.path.join(self.image_path, sample)).convert(
                    "RGB",
                )
                for sample in self.samples
            ]

    def __getitem__(self, index: int):
        """Returns a sample from the dataset.

        Args:
            idx (_type_): index of the sample we want to retrieve

        Returns
        -------
            _type_: sample we want to retrieve

        """
        if self.debug:
            img = Image.open(os.path.join(self.image_path, self.samples[index])).convert(
                "RGB",
            )
        else:
            img = self.images[index]

        if self.transform:
            img = self.transform(img)

        return (
            img,
            self.sensitive_attributes[index],
            self.targets[index],
            self.indexes[index],
            index,
        )

    def __len__(self) -> int:
        """This function returns the size of the dataset.

        Returns
        -------
            int: size of the dataset
        """
        return self.n_samples

In [12]:
import torch.nn as nn
from torch import Tensor, nn


class CelebaNetHair(nn.Module):
    """This class defines the CelebaNet."""

    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 4,
        dropout_rate: float = 0,
    ) -> None:
        """Initializes the CelebaNet network.

        Args:
        ----
            in_channels (int, optional): Number of input channels . Defaults to 3.
            num_classes (int, optional): Number of classes . Defaults to 2.
            dropout_rate (float, optional): _description_. Defaults to 0.2.
        """
        super().__init__()
        self.cnn1 = nn.Conv2d(
            in_channels,
            8,
            kernel_size=(3, 3),
            padding=(1, 1),
            stride=(1, 1),
        )
        self.cnn2 = nn.Conv2d(8, 16, kernel_size=(3, 3), padding=(1, 1), stride=(1, 1))
        self.cnn3 = nn.Conv2d(16, 32, kernel_size=(3, 3), padding=(1, 1), stride=(1, 1))
        self.fc1 = nn.Linear(2048, 2)
        self.gn_relu = nn.Sequential(
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
        )
        # self.dropout = nn.Dropout(dropout_rate)

    def forward(self, input_data: Tensor) -> Tensor:
        """Defines the forward pass of the network.

        Args:
            input_data (Tensor): Input data

        Returns
        -------
            Tensor: Output data
        """
        out = self.gn_relu(self.cnn1(input_data))
        out = self.gn_relu(self.cnn2(out))
        out = self.gn_relu(self.cnn3(out))
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        return out

In [13]:
transform = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ],
)

In [14]:
train_dataset = CelebaDatasetHairColor(
    csv_path="../../../data/celeba/train_with_validation_hair.csv",
    image_path="../../../data/img_align_celeba/img_align_celeba",
    transform=transform,
    debug=True,
)
validation_dataset = CelebaDatasetHairColor(
    csv_path="../../../data/celeba/validation_hair.csv",
    image_path="../../../data/img_align_celeba/img_align_celeba",
    transform=transform,
    debug=True,
)
test_dataset = CelebaDatasetHairColor(
    csv_path="../../../data/celeba/test_hair.csv",
    image_path="../../../data/img_align_celeba/img_align_celeba",
    transform=transform,
    debug=True,
)

In [19]:
Counter(train_dataset.targets)

Counter({0: 60483, 1: 18026})

In [20]:
Counter(validation_dataset.targets)

Counter({0: 15043, 1: 4584})

In [21]:
Counter(test_dataset.targets)

Counter({0: 18892, 1: 5642})

In [15]:
np.unique(train_dataset.targets)

array([0, 1])

In [16]:
np.unique(train_dataset.sensitive_attributes)

array([0, 1])

In [17]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True

batch_size = 128
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = CelebaNetHair()
lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
epochs = 2
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [18]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)


validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)